#LLM HALLUCINATION ANALYZER.

#PHASE 1: PROJECT STRUCTURE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
mkdir -p /content/drive/MyDrive/hallucination-analyzer


In [ ]:
%%bash
cd /content/drive/MyDrive/

mkdir -p hallucination-analyzer/app/services
mkdir -p hallucination-analyzer/app/utils
mkdir -p hallucination-analyzer/data
mkdir -p hallucination-analyzer/notebooks

touch hallucination-analyzer/app/main.py
touch hallucination-analyzer/app/config.py
touch hallucination-analyzer/app/schemas.py

touch hallucination-analyzer/app/services/nli_checker.py
touch hallucination-analyzer/app/services/similarity_checker.py
touch hallucination-analyzer/app/services/scorer.py
touch hallucination-analyzer/app/services/claim_splitter.py
touch hallucination-analyzer/app/services/pipeline.py

touch hallucination-analyzer/app/utils/text_cleaner.py
touch hallucination-analyzer/app/utils/helpers.py

touch hallucination-analyzer/data/sample_cases.json
touch hallucination-analyzer/requirements.txt
touch hallucination-analyzer/README.md


In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer
find . -type f


./app/services/nli_checker.py
./app/services/similarity_checker.py
./app/services/scorer.py
./app/services/claim_splitter.py
./app/services/pipeline.py
./app/utils/text_cleaner.py
./app/utils/helpers.py
./app/__pycache__/config.cpython-312.pyc
./app/main.py
./app/config.py
./app/schemas.py
./data/sample_cases.json
./requirements.txt
./README.md


In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > requirements.txt
torch
transformers
sentence-transformers
scikit-learn
numpy
pandas
fastapi
uvicorn
python-multipart
EOL


In [ ]:
!pip install -r /content/drive/MyDrive/hallucination-analyzer/requirements.txt


In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/config.py
# app/config.py

# ----------------------------
# Model Names
# ----------------------------

# Using RoBERTa MNLI because it is fully public + stable on Colab
NLI_MODEL_NAME = "roberta-large-mnli"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

# ----------------------------
# Hybrid Scoring Weights
# ----------------------------

NLI_WEIGHT = 0.65
SIMILARITY_WEIGHT = 0.35

# ----------------------------
# Classification Threshold
# ----------------------------

SUPPORTED_THRESHOLD = 0.60

# ----------------------------
# Labels
# ----------------------------

LABEL_SUPPORTED = "SUPPORTED"
LABEL_NOT_SUPPORTED = "NOT_SUPPORTED"

# NLI Labels (standard MNLI output)
LABEL_ENTAILMENT = "ENTAILMENT"
LABEL_NEUTRAL = "NEUTRAL"
LABEL_CONTRADICTION = "CONTRADICTION"
EOL


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/hallucination-analyzer")

from app.config import NLI_MODEL_NAME, EMBEDDING_MODEL_NAME

print("NLI Model:", NLI_MODEL_NAME)
print("Embedding Model:", EMBEDDING_MODEL_NAME)


NLI Model: roberta-large-mnli
Embedding Model: sentence-transformers/all-mpnet-base-v2


#PHASE 2: Build the NLI Checker Module (nli_checker.py)

This module will:
✅ Load DeBERTa MNLI
✅ Take (context, answer)
✅ Return:

entailment probability

neutral probability

contradiction probability

predicted label

In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/services/nli_checker.py
# app/services/nli_checker.py

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from app.config import NLI_MODEL_NAME


class NLIChecker:
    """
    This class checks whether a hypothesis (answer) is supported by a premise (context)
    using a pretrained NLI model.
    """

    def __init__(self, model_name: str = NLI_MODEL_NAME, device: str = None):
        self.model_name = model_name

        # Auto-detect device
        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device

        # Load tokenizer + model
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)

        self.model.to(self.device)
        self.model.eval()

        # For MNLI models, label mapping is usually:
        # 0 = contradiction, 1 = neutral, 2 = entailment
        self.label_map = {0: "CONTRADICTION", 1: "NEUTRAL", 2: "ENTAILMENT"}

    def predict(self, premise: str, hypothesis: str):
        """
        Returns NLI classification probabilities for:
        - entailment
        - neutral
        - contradiction
        """

        if not premise or not hypothesis:
            raise ValueError("Premise and hypothesis cannot be empty.")

        # Tokenize input
        inputs = self.tokenizer(
            premise,
            hypothesis,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Run inference
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits

            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

        contradiction_prob = float(probs[0])
        neutral_prob = float(probs[1])
        entailment_prob = float(probs[2])

        predicted_class = int(probs.argmax())
        predicted_label = self.label_map[predicted_class]

        return {
            "label": predicted_label,
            "entailment_prob": entailment_prob,
            "neutral_prob": neutral_prob,
            "contradiction_prob": contradiction_prob
        }
EOL


In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

from app.services.nli_checker import NLIChecker

nli = NLIChecker()

context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "The capital of Punjab is Chandigarh."

result = nli.predict(context, answer)

print(result)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'label': 'ENTAILMENT', 'entailment_prob': 0.9942728877067566, 'neutral_prob': 0.004246322438120842, 'contradiction_prob': 0.001480724080465734}


In [ ]:
context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "The capital of Punjab is Mumbai."

result = nli.predict(context, answer)
print(result)


{'label': 'CONTRADICTION', 'entailment_prob': 0.0002238274464616552, 'neutral_prob': 0.0006977049633860588, 'contradiction_prob': 0.9990785121917725}


#PHASE 3: Semantic Similarity Module (SBERT + Cosine Similarity)

This module will:
* ✅ load SentenceTransformer model
* ✅ embed context + answer
* ✅ compute cosine similarity
* ✅ return similarity score (0–1)

This becomes the 2nd brain of hallucination detection.

In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/services/similarity_checker.py
# app/services/similarity_checker.py

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from app.config import EMBEDDING_MODEL_NAME


class SimilarityChecker:
    """
    Computes semantic similarity between context and answer
    using Sentence-BERT embeddings.
    """

    def __init__(self, model_name: str = EMBEDDING_MODEL_NAME):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)

    def compute_similarity(self, context: str, answer: str) -> float:
        """
        Returns cosine similarity score between context and answer.
        Range: 0 to 1 (usually)
        """

        if not context or not answer:
            raise ValueError("Context and answer cannot be empty.")

        embeddings = self.model.encode([context, answer])

        context_emb = embeddings[0].reshape(1, -1)
        answer_emb = embeddings[1].reshape(1, -1)

        similarity = cosine_similarity(context_emb, answer_emb)[0][0]

        return float(similarity)
EOL


In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

from app.services.similarity_checker import SimilarityChecker

sim_checker = SimilarityChecker()

context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "Chandigarh is the capital of Punjab."

score = sim_checker.compute_similarity(context, answer)
print("Similarity Score:", score)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity Score: 0.889894962310791


In [ ]:
context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "Tokyo is the capital of Japan."

score = sim_checker.compute_similarity(context, answer)
print("Similarity Score:", score)


Similarity Score: 0.23390771448612213


# PHASE 4: Hybrid Scoring + Final Decision Module

This step will create:

app/services/scorer.py

It will:
* ✅ take NLI output
* ✅ take similarity score
* ✅ compute final confidence
* ✅ decide SUPPORTED / NOT_SUPPORTED

In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/services/scorer.py
# app/services/scorer.py

from app.config import (
    NLI_WEIGHT,
    SIMILARITY_WEIGHT,
    SUPPORTED_THRESHOLD,
    LABEL_SUPPORTED,
    LABEL_NOT_SUPPORTED
)


class HybridScorer:
    """
    Combines NLI entailment probability + semantic similarity
    to compute final confidence score and classification.
    """

    def __init__(
        self,
        nli_weight: float = NLI_WEIGHT,
        similarity_weight: float = SIMILARITY_WEIGHT,
        threshold: float = SUPPORTED_THRESHOLD
    ):
        if abs(nli_weight + similarity_weight - 1.0) > 0.001:
            raise ValueError("Weights must sum to 1.0")

        self.nli_weight = nli_weight
        self.similarity_weight = similarity_weight
        self.threshold = threshold

    def compute_score(self, entailment_prob: float, similarity_score: float) -> float:
        """
        Computes final hybrid score (0 to 1).
        """
        final_score = (
            self.nli_weight * entailment_prob +
            self.similarity_weight * similarity_score
        )
        return float(final_score)

    def classify(self, entailment_prob: float, similarity_score: float):
        """
        Returns final label + confidence score.
        """

        final_score = self.compute_score(entailment_prob, similarity_score)

        if final_score >= self.threshold:
            final_label = LABEL_SUPPORTED
        else:
            final_label = LABEL_NOT_SUPPORTED

        return {
            "final_label": final_label,
            "confidence": final_score
        }
EOL


In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

from app.services.scorer import HybridScorer

scorer = HybridScorer()

# Example: high entailment + high similarity
print(scorer.classify(entailment_prob=0.92, similarity_score=0.85))

# Example: low entailment + medium similarity
print(scorer.classify(entailment_prob=0.15, similarity_score=0.55))


{'final_label': 'SUPPORTED', 'confidence': 0.8955000000000001}
{'final_label': 'NOT_SUPPORTED', 'confidence': 0.29000000000000004}


In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

from app.services.nli_checker import NLIChecker
from app.services.similarity_checker import SimilarityChecker
from app.services.scorer import HybridScorer

nli = NLIChecker()
sim = SimilarityChecker()
scorer = HybridScorer()

context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "Chandigarh is the capital of Punjab."

nli_result = nli.predict(context, answer)
similarity_score = sim.compute_similarity(context, answer)

final_result = scorer.classify(
    entailment_prob=nli_result["entailment_prob"],
    similarity_score=similarity_score
)

print("NLI Result:", nli_result)
print("Similarity Score:", similarity_score)
print("Final Result:", final_result)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI Result: {'label': 'ENTAILMENT', 'entailment_prob': 0.9941293001174927, 'neutral_prob': 0.004298721440136433, 'contradiction_prob': 0.001572011155076325}
Similarity Score: 0.889894962310791
Final Result: {'final_label': 'SUPPORTED', 'confidence': 0.9576472818851471}


In [ ]:
context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "Mumbai is the capital of Punjab."

nli_result = nli.predict(context, answer)
similarity_score = sim.compute_similarity(context, answer)

final_result = scorer.classify(
    entailment_prob=nli_result["entailment_prob"],
    similarity_score=similarity_score
)

print("NLI Result:", nli_result)
print("Similarity Score:", similarity_score)
print("Final Result:", final_result)


NLI Result: {'label': 'CONTRADICTION', 'entailment_prob': 0.0005894293426536024, 'neutral_prob': 0.0016245320439338684, 'contradiction_prob': 0.9977860450744629}
Similarity Score: 0.6159805655479431
Final Result: {'final_label': 'NOT_SUPPORTED', 'confidence': 0.21597632701450492}


#PHASE 5: Full Pipeline Module (pipeline.py)

This is the heart of the project.

It will:
* ✅ take context + answer
* ✅ call NLI checker
* ✅ call similarity checker
* ✅ call scorer
* ✅ return final structured JSON output

In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/services/pipeline.py
# app/services/pipeline.py

from app.services.nli_checker import NLIChecker
from app.services.similarity_checker import SimilarityChecker
from app.services.scorer import HybridScorer


class HallucinationPipeline:
    """
    Main pipeline that runs:
    1. NLI inference
    2. Semantic similarity
    3. Hybrid scoring
    """

    def __init__(self):
        self.nli_checker = NLIChecker()
        self.similarity_checker = SimilarityChecker()
        self.scorer = HybridScorer()

    def analyze(self, context: str, answer: str):
        """
        Returns full hallucination analysis output.
        """

        if not context or not answer:
            raise ValueError("Context and answer cannot be empty.")

        # Step 1: NLI
        nli_result = self.nli_checker.predict(context, answer)

        # Step 2: Similarity
        similarity_score = self.similarity_checker.compute_similarity(context, answer)

        # Step 3: Hybrid Score + Final Classification
        final_result = self.scorer.classify(
            entailment_prob=nli_result["entailment_prob"],
            similarity_score=similarity_score
        )

        return {
            "final_label": final_result["final_label"],
            "confidence": final_result["confidence"],
            "nli": nli_result,
            "semantic_similarity": similarity_score
        }
EOL


In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

from app.services.pipeline import HallucinationPipeline

pipeline = HallucinationPipeline()

context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "Chandigarh is the capital of Punjab."

result = pipeline.analyze(context, answer)
print(result)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'final_label': 'SUPPORTED', 'confidence': 0.9576472818851471, 'nli': {'label': 'ENTAILMENT', 'entailment_prob': 0.9941293001174927, 'neutral_prob': 0.004298721440136433, 'contradiction_prob': 0.001572011155076325}, 'semantic_similarity': 0.889894962310791}


In [ ]:
context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "Mumbai is the capital of Punjab."

result = pipeline.analyze(context, answer)
print(result)


{'final_label': 'NOT_SUPPORTED', 'confidence': 0.21597632701450492, 'nli': {'label': 'CONTRADICTION', 'entailment_prob': 0.0005894293426536024, 'neutral_prob': 0.0016245320439338684, 'contradiction_prob': 0.9977860450744629}, 'semantic_similarity': 0.6159805655479431}


#PHASE 6: FastAPI Backend (Deployment Ready)

Now we’ll create:

schemas.py (request/response format)

main.py (API server)

So your project becomes a proper LLM hallucination detection API.

In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/schemas.py
# app/schemas.py

from pydantic import BaseModel


class HallucinationRequest(BaseModel):
    context: str
    answer: str


class HallucinationResponse(BaseModel):
    final_label: str
    confidence: float
    semantic_similarity: float
    nli: dict
EOL


In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/main.py
# app/main.py

from fastapi import FastAPI
from app.schemas import HallucinationRequest, HallucinationResponse
from app.services.pipeline import HallucinationPipeline

app = FastAPI(
    title="LLM Hallucination Analyzer",
    description="Grounded Response Verification System using NLI + Semantic Similarity",
    version="1.0"
)

pipeline = HallucinationPipeline()


@app.get("/")
def home():
    return {"message": "LLM Hallucination Analyzer API is running"}


@app.post("/analyze", response_model=HallucinationResponse)
def analyze_hallucination(request: HallucinationRequest):
    result = pipeline.analyze(request.context, request.answer)
    return result
EOL


In [ ]:
%cd /content/drive/MyDrive/hallucination-analyzer
!nohup uvicorn app.main:app --host 0.0.0.0 --port 8000 &


/content/drive/MyDrive/hallucination-analyzer
nohup: appending output to 'nohup.out'


In [ ]:
!ps aux | grep uvicorn


root       12457 63.1  2.7 910104 370188 ?       Rl   16:11   0:10 /usr/bin/python3 /usr/local/bin/uvicorn app.main:app --host 0.0.0.0 --port 8000
root       12528  0.0  0.0   7372  3464 ?        S    16:11   0:00 /bin/bash -c ps aux | grep uvicorn
root       12530  0.0  0.0   6480  2524 ?        S    16:11   0:00 grep uvicorn


In [ ]:
import requests

url = "http://127.0.0.1:8000/analyze"

payload = {
    "context": "Punjab is a state in India. Chandigarh is the capital of Punjab.",
    "answer": "Chandigarh is the capital of Punjab."
}

res = requests.post(url, json=payload)
print(res.status_code)
print(res.json())


200
{'final_label': 'SUPPORTED', 'confidence': 0.9576472818851471, 'semantic_similarity': 0.889894962310791, 'nli': {'label': 'ENTAILMENT', 'entailment_prob': 0.9941293001174927, 'neutral_prob': 0.004298721440136433, 'contradiction_prob': 0.001572011155076325}}


#PHASE 7: Claim-Level Hallucination Detection (Multi-sentence verification)

Because currently we check the whole answer at once.
But in reality, answers contain multiple claims.

So we’ll do:

Example Answer:

"Chandigarh is the capital of Punjab. Punjab is the richest state."

Split into:

Chandigarh is the capital of Punjab.

Punjab is the richest state.

Then verify each claim independently.

🧠 How Claim-Level Pipeline Works
Input:

Context + Answer

Output:

claim wise results

final aggregated result



In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/services/claim_splitter.py
# app/services/claim_splitter.py

import re


class ClaimSplitter:
    """
    Splits an answer into individual claims (sentences).
    Simple rule-based splitter.
    """

    def split_into_claims(self, text: str):
        if not text or not text.strip():
            return []

        # Clean extra spaces
        text = re.sub(r"\s+", " ", text.strip())

        # Split by sentence endings
        claims = re.split(r"(?<=[.!?])\s+", text)

        # Remove empty claims
        claims = [c.strip() for c in claims if c.strip()]

        return claims
EOL


__UPDATING PIPELINE TO SUPPORT MULTI CLAIM MODE__

In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/services/pipeline.py
# app/services/pipeline.py

from app.services.nli_checker import NLIChecker
from app.services.similarity_checker import SimilarityChecker
from app.services.scorer import HybridScorer
from app.services.claim_splitter import ClaimSplitter

from app.config import LABEL_SUPPORTED, LABEL_NOT_SUPPORTED


class HallucinationPipeline:
    """
    Main pipeline that runs:
    1. NLI inference
    2. Semantic similarity
    3. Hybrid scoring
    Supports:
    - Full answer verification
    - Claim-wise verification
    """

    def __init__(self):
        self.nli_checker = NLIChecker()
        self.similarity_checker = SimilarityChecker()
        self.scorer = HybridScorer()
        self.claim_splitter = ClaimSplitter()

    def analyze(self, context: str, answer: str):
        """
        Standard analysis: checks full answer at once.
        """

        if not context or not answer:
            raise ValueError("Context and answer cannot be empty.")

        nli_result = self.nli_checker.predict(context, answer)
        similarity_score = self.similarity_checker.compute_similarity(context, answer)

        final_result = self.scorer.classify(
            entailment_prob=nli_result["entailment_prob"],
            similarity_score=similarity_score
        )

        return {
            "mode": "full_answer",
            "final_label": final_result["final_label"],
            "confidence": final_result["confidence"],
            "nli": nli_result,
            "semantic_similarity": similarity_score
        }

    def analyze_claims(self, context: str, answer: str):
        """
        Claim-wise analysis: splits answer into claims and checks each separately.
        Aggregates results.
        """

        if not context or not answer:
            raise ValueError("Context and answer cannot be empty.")

        claims = self.claim_splitter.split_into_claims(answer)

        if len(claims) == 0:
            raise ValueError("No valid claims found in answer.")

        claim_results = []
        supported_count = 0
        total_confidence = 0.0

        for claim in claims:
            nli_result = self.nli_checker.predict(context, claim)
            similarity_score = self.similarity_checker.compute_similarity(context, claim)

            final_result = self.scorer.classify(
                entailment_prob=nli_result["entailment_prob"],
                similarity_score=similarity_score
            )

            if final_result["final_label"] == LABEL_SUPPORTED:
                supported_count += 1

            total_confidence += final_result["confidence"]

            claim_results.append({
                "claim": claim,
                "final_label": final_result["final_label"],
                "confidence": final_result["confidence"],
                "nli": nli_result,
                "semantic_similarity": similarity_score
            })

        avg_confidence = total_confidence / len(claims)

        # Aggregation logic:
        # If any claim is NOT_SUPPORTED, final is NOT_SUPPORTED
        overall_label = LABEL_SUPPORTED
        for cr in claim_results:
            if cr["final_label"] == LABEL_NOT_SUPPORTED:
                overall_label = LABEL_NOT_SUPPORTED
                break

        return {
            "mode": "claim_wise",
            "final_label": overall_label,
            "confidence": avg_confidence,
            "total_claims": len(claims),
            "supported_claims": supported_count,
            "claims": claim_results
        }
EOL


In [ ]:
import sys
import importlib

sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

import app.services.pipeline
importlib.reload(app.services.pipeline)

from app.services.pipeline import HallucinationPipeline

pipeline = HallucinationPipeline()

print(dir(pipeline))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'analyze', 'analyze_claims', 'claim_splitter', 'nli_checker', 'scorer', 'similarity_checker']


In [ ]:
context = "Punjab is a state in India. Chandigarh is the capital of Punjab."
answer = "Chandigarh is the capital of Punjab. Mumbai is the capital of Punjab."

result = pipeline.analyze_claims(context, answer)
print(result)


{'mode': 'claim_wise', 'final_label': 'NOT_SUPPORTED', 'confidence': 0.5868118044498261, 'total_claims': 2, 'supported_claims': 1, 'claims': [{'claim': 'Chandigarh is the capital of Punjab.', 'final_label': 'SUPPORTED', 'confidence': 0.9576472818851471, 'nli': {'label': 'ENTAILMENT', 'entailment_prob': 0.9941293001174927, 'neutral_prob': 0.004298721440136433, 'contradiction_prob': 0.001572011155076325}, 'semantic_similarity': 0.889894962310791}, {'claim': 'Mumbai is the capital of Punjab.', 'final_label': 'NOT_SUPPORTED', 'confidence': 0.21597632701450492, 'nli': {'label': 'CONTRADICTION', 'entailment_prob': 0.0005894293426536024, 'neutral_prob': 0.0016245320439338684, 'contradiction_prob': 0.9977860450744629}, 'semantic_similarity': 0.6159805655479431}]}


In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > app/main.py
# app/main.py

from fastapi import FastAPI
from app.schemas import HallucinationRequest
from app.services.pipeline import HallucinationPipeline

app = FastAPI(
    title="LLM Hallucination Analyzer",
    description="Grounded Response Verification System using NLI + Semantic Similarity",
    version="2.0"
)

pipeline = HallucinationPipeline()


@app.get("/")
def home():
    return {"message": "LLM Hallucination Analyzer API is running"}


@app.post("/analyze")
def analyze_full_answer(request: HallucinationRequest):
    return pipeline.analyze(request.context, request.answer)


@app.post("/analyze-claims")
def analyze_claims(request: HallucinationRequest):
    return pipeline.analyze_claims(request.context, request.answer)
EOL


In [ ]:
!ps aux | grep uvicorn


root       14064  4.9  4.4 3398960 589192 ?      Sl   16:17   0:12 /usr/bin/python3 /usr/local/bin/uvicorn app.main:app --host 0.0.0.0 --port 8000
root       15126  0.0  0.0   7372  3480 ?        S    16:22   0:00 /bin/bash -c ps aux | grep uvicorn
root       15128  0.0  0.0   6480  2424 ?        S    16:22   0:00 grep uvicorn


In [ ]:
!pkill -f uvicorn


In [ ]:
%cd /content/drive/MyDrive/hallucination-analyzer
!nohup uvicorn app.main:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &


/content/drive/MyDrive/hallucination-analyzer


In [ ]:
!cat /content/drive/MyDrive/hallucination-analyzer/server.log

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1451.47it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]
RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1435.42it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:     Started server process [15252]
INFO:     Waitin

In [ ]:
import requests
print(requests.get("http://127.0.0.1:8000/").json())

{'message': 'LLM Hallucination Analyzer API is running'}


In [ ]:
import requests

url = "http://127.0.0.1:8000/analyze-claims"

payload = {
    "context": "Punjab is a state in India. Chandigarh is the capital of Punjab.",
    "answer": "Chandigarh is the capital of Punjab. Mumbai is the capital of Punjab."
}

res = requests.post(url, json=payload)
print(res.status_code)
print(res.json())


200
{'mode': 'claim_wise', 'final_label': 'NOT_SUPPORTED', 'confidence': 0.5868118044498261, 'total_claims': 2, 'supported_claims': 1, 'claims': [{'claim': 'Chandigarh is the capital of Punjab.', 'final_label': 'SUPPORTED', 'confidence': 0.9576472818851471, 'nli': {'label': 'ENTAILMENT', 'entailment_prob': 0.9941293001174927, 'neutral_prob': 0.004298721440136433, 'contradiction_prob': 0.001572011155076325}, 'semantic_similarity': 0.889894962310791}, {'claim': 'Mumbai is the capital of Punjab.', 'final_label': 'NOT_SUPPORTED', 'confidence': 0.21597632701450492, 'nli': {'label': 'CONTRADICTION', 'entailment_prob': 0.0005894293426536024, 'neutral_prob': 0.0016245320439338684, 'contradiction_prob': 0.9977860450744629}, 'semantic_similarity': 0.6159805655479431}]}


#Dataset Testing + Evaluation (Metrics)
__A)SELF MADE MINI DATASET__

We’ll do this properly:
* ✅ create a mini dataset (20-50 samples)
* ✅ run both endpoints (full + claimwise)
* ✅ calculate Accuracy / Precision / Recall / F1
* ✅ confusion matrix

In [ ]:
%%bash
cd /content/drive/MyDrive/hallucination-analyzer

cat <<EOL > data/sample_cases.json
[
  {
    "context": "Punjab is a state in India. Chandigarh is the capital of Punjab.",
    "answer": "Chandigarh is the capital of Punjab.",
    "label": "SUPPORTED"
  },
  {
    "context": "Punjab is a state in India. Chandigarh is the capital of Punjab.",
    "answer": "Mumbai is the capital of Punjab.",
    "label": "NOT_SUPPORTED"
  },
  {
    "context": "The Earth revolves around the Sun. It takes about 365 days to complete one orbit.",
    "answer": "The Earth takes 365 days to revolve around the Sun.",
    "label": "SUPPORTED"
  },
  {
    "context": "The Earth revolves around the Sun. It takes about 365 days to complete one orbit.",
    "answer": "The Earth takes 200 days to revolve around the Sun.",
    "label": "NOT_SUPPORTED"
  },
  {
    "context": "Apple is a technology company founded by Steve Jobs, Steve Wozniak, and Ronald Wayne.",
    "answer": "Apple was founded by Steve Jobs and Steve Wozniak.",
    "label": "SUPPORTED"
  },
  {
    "context": "Apple is a technology company founded by Steve Jobs, Steve Wozniak, and Ronald Wayne.",
    "answer": "Apple was founded by Elon Musk.",
    "label": "NOT_SUPPORTED"
  },
  {
    "context": "Water boils at 100 degrees Celsius at sea level.",
    "answer": "Water boils at 100 degrees Celsius at sea level.",
    "label": "SUPPORTED"
  },
  {
    "context": "Water boils at 100 degrees Celsius at sea level.",
    "answer": "Water boils at 50 degrees Celsius at sea level.",
    "label": "NOT_SUPPORTED"
  },
  {
    "context": "Virat Kohli is an Indian cricketer known for his batting.",
    "answer": "Virat Kohli is known for his batting.",
    "label": "SUPPORTED"
  },
  {
    "context": "Virat Kohli is an Indian cricketer known for his batting.",
    "answer": "Virat Kohli is a footballer.",
    "label": "NOT_SUPPORTED"
  }
]
EOL


In [ ]:
import json
import sys
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

from app.services.pipeline import HallucinationPipeline

pipeline = HallucinationPipeline()

# Load dataset
with open("/content/drive/MyDrive/hallucination-analyzer/data/sample_cases.json", "r") as f:
    dataset = json.load(f)

y_true = []
y_pred = []
rows = []

for item in dataset:
    context = item["context"]
    answer = item["answer"]
    true_label = item["label"]

    result = pipeline.analyze(context, answer)
    predicted_label = result["final_label"]
    confidence = result["confidence"]

    y_true.append(true_label)
    y_pred.append(predicted_label)

    rows.append({
        "context": context,
        "answer": answer,
        "true_label": true_label,
        "predicted_label": predicted_label,
        "confidence": confidence
    })

df = pd.DataFrame(rows)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

df


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Accuracy: 1.0

Classification Report:

               precision    recall  f1-score   support

NOT_SUPPORTED       1.00      1.00      1.00         5
    SUPPORTED       1.00      1.00      1.00         5

     accuracy                           1.00        10
    macro avg       1.00      1.00      1.00        10
 weighted avg       1.00      1.00      1.00        10


Confusion Matrix:

[[5 0]
 [0 5]]


,context,answer,true_label,predicted_label,confidence
0,Punjab is a state in India. Chandigarh is the ...,Chandigarh is the capital of Punjab.,SUPPORTED,SUPPORTED,0.957647
1,Punjab is a state in India. Chandigarh is the ...,Mumbai is the capital of Punjab.,NOT_SUPPORTED,NOT_SUPPORTED,0.215976
2,The Earth revolves around the Sun. It takes ab...,The Earth takes 365 days to revolve around the...,SUPPORTED,SUPPORTED,0.947896
3,The Earth revolves around the Sun. It takes ab...,The Earth takes 200 days to revolve around the...,NOT_SUPPORTED,NOT_SUPPORTED,0.289091
4,Apple is a technology company founded by Steve...,Apple was founded by Steve Jobs and Steve Wozn...,SUPPORTED,SUPPORTED,0.931224
5,Apple is a technology company founded by Steve...,Apple was founded by Elon Musk.,NOT_SUPPORTED,NOT_SUPPORTED,0.258308
6,Water boils at 100 degrees Celsius at sea level.,Water boils at 100 degrees Celsius at sea level.,SUPPORTED,SUPPORTED,0.996034
7,Water boils at 100 degrees Celsius at sea level.,Water boils at 50 degrees Celsius at sea level.,NOT_SUPPORTED,NOT_SUPPORTED,0.324888
8,Virat Kohli is an Indian cricketer known for h...,Virat Kohli is known for his batting.,SUPPORTED,SUPPORTED,0.966864
9,Virat Kohli is an Indian cricketer known for h...,Virat Kohli is a footballer.,NOT_SUPPORTED,NOT_SUPPORTED,0.303859


In [ ]:
y_true = []
y_pred = []
rows = []

for item in dataset:
    context = item["context"]
    answer = item["answer"]
    true_label = item["label"]

    result = pipeline.analyze_claims(context, answer)
    predicted_label = result["final_label"]
    confidence = result["confidence"]

    y_true.append(true_label)
    y_pred.append(predicted_label)

    rows.append({
        "answer": answer,
        "true_label": true_label,
        "predicted_label": predicted_label,
        "confidence": confidence,
        "total_claims": result["total_claims"],
        "supported_claims": result["supported_claims"]
    })

df_claim = pd.DataFrame(rows)

print("Claim-wise Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report (Claim-wise):\n")
print(classification_report(y_true, y_pred))

df_claim


Claim-wise Accuracy: 1.0

Classification Report (Claim-wise):

               precision    recall  f1-score   support

NOT_SUPPORTED       1.00      1.00      1.00         5
    SUPPORTED       1.00      1.00      1.00         5

     accuracy                           1.00        10
    macro avg       1.00      1.00      1.00        10
 weighted avg       1.00      1.00      1.00        10



,answer,true_label,predicted_label,confidence,total_claims,supported_claims
0,Chandigarh is the capital of Punjab.,SUPPORTED,SUPPORTED,0.957647,1,1
1,Mumbai is the capital of Punjab.,NOT_SUPPORTED,NOT_SUPPORTED,0.215976,1,0
2,The Earth takes 365 days to revolve around the...,SUPPORTED,SUPPORTED,0.947896,1,1
3,The Earth takes 200 days to revolve around the...,NOT_SUPPORTED,NOT_SUPPORTED,0.289091,1,0
4,Apple was founded by Steve Jobs and Steve Wozn...,SUPPORTED,SUPPORTED,0.931224,1,1
5,Apple was founded by Elon Musk.,NOT_SUPPORTED,NOT_SUPPORTED,0.258308,1,0
6,Water boils at 100 degrees Celsius at sea level.,SUPPORTED,SUPPORTED,0.996034,1,1
7,Water boils at 50 degrees Celsius at sea level.,NOT_SUPPORTED,NOT_SUPPORTED,0.324888,1,0
8,Virat Kohli is known for his batting.,SUPPORTED,SUPPORTED,0.966864,1,1
9,Virat Kohli is a footballer.,NOT_SUPPORTED,NOT_SUPPORTED,0.303859,1,0


#PHASE 9: FEVER Evaluation + Tuning + Graphs

We’ll do in this order:

* 1️⃣ Load FEVER
* 2️⃣ Convert it into (context, claim, label)
* 3️⃣ Run predictions on 500–2000 samples
* 4️⃣ Evaluate accuracy/F1
* 5️⃣ Tune threshold + weights
* 6️⃣ Plot confidence distribution

In [ ]:
!pip install datasets -q


In [ ]:
from datasets import load_dataset

fever = load_dataset("copenlu/fever_gold_evidence")
print(fever)


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/96.9M [00:00<?, ?B/s]

valid.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/228277 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15935 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/16039 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['claim', 'label', 'evidence', 'id', 'verifiable', 'original_id'],
        num_rows: 228277
    })
    validation: Dataset({
        features: ['claim', 'label', 'evidence', 'id', 'verifiable', 'original_id'],
        num_rows: 15935
    })
    test: Dataset({
        features: ['claim', 'label', 'evidence', 'id', 'verifiable', 'original_id'],
        num_rows: 16039
    })
})


In [ ]:
print(fever["train"].column_names)


['claim', 'label', 'evidence', 'id', 'verifiable', 'original_id']


In [ ]:
print(fever["train"][0])


{'claim': 'The number of new cases of shingles per year extends from 1.2–3.4 per 1,000 among healthy individuals.', 'label': 'SUPPORTS', 'evidence': [['Shingles', '31', 'The number of new cases per year ranges from 1.2 -- 3.4 per 1,000 among healthy individuals to 3.9 -- 11.8 per 1,000 among those older than 65 years of age .']], 'id': '98faa551d5973b62591e0835bf898d84', 'verifiable': 'VERIFIABLE', 'original_id': 123397}


In [ ]:
import pandas as pd

df_temp = pd.DataFrame(fever["train"][:20])
df_temp.head()


,claim,label,evidence,id,verifiable,original_id
0,The number of new cases of shingles per year e...,SUPPORTS,"[[Shingles, 31, The number of new cases per ye...",98faa551d5973b62591e0835bf898d84,VERIFIABLE,123397
1,Gabrielle Union was in a movie.,SUPPORTS,"[[Gabrielle_Union, 8, She co-starred in film T...",7a332faa8aee470aa820506ccc38063e,VERIFIABLE,930
2,Eleveneleven was founded by a chef.,REFUTES,"[[Eleveneleven, 0, eleveneleven is a record la...",c457e9fef2f45fc423cbdb67ff321f67,VERIFIABLE,213765
3,Cosmos: A Spacetime Odyssey secured studio sup...,NOT ENOUGH INFO,"[[Seth_MacFarlane, 21, MacFarlane served as ex...",3adeacf2a6e566c56d0e59b87a9b5ba2,NOT VERIFIABLE,127497
4,The World According to Paris starred Hilton's ...,NOT ENOUGH INFO,"[[The_World_According_to_Paris, 4, It was film...",9c8a6e97bdaff4043800b19bff223425,NOT VERIFIABLE,78717


In [ ]:
import random

def build_fever_samples(ds, sample_size=500, include_nei=False):
    samples = []
    indices = list(range(len(ds)))
    random.shuffle(indices)

    for idx in indices:
        row = ds[idx]

        claim = row["claim"]
        label = row["label"]
        evidence_list = row["evidence"]

        # Skip NOT ENOUGH INFO unless we want it
        if label == "NOT ENOUGH INFO" and not include_nei:
            continue

        # Extract evidence sentences (3rd element in each evidence item)
        context_sentences = []
        for ev_group in evidence_list:
            if len(ev_group) >= 3:
                sentence = ev_group[2]
                if sentence:
                    context_sentences.append(sentence)

        if len(context_sentences) == 0:
            continue

        context = " ".join(context_sentences)

        # Map labels to our binary system
        if label == "SUPPORTS":
            mapped_label = "SUPPORTED"
        else:
            mapped_label = "NOT_SUPPORTED"  # REFUTES + NEI

        samples.append({
            "context": context,
            "answer": claim,
            "label": mapped_label,
            "original_label": label
        })

        if len(samples) >= sample_size:
            break

    return samples


fever_samples = build_fever_samples(fever["train"], sample_size=500, include_nei=False)

print("Prepared samples:", len(fever_samples))
print(fever_samples[0])


Prepared samples: 500
{'context': 'Cena has the fourth-highest number of combined days as WWE Champion , behind Bruno Sammartino , Bob Backlund , and Hulk Hogan .', 'answer': 'John Cena has won championships.', 'label': 'SUPPORTED', 'original_label': 'SUPPORTS'}


In [ ]:
import sys
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

sys.path.insert(0, "/content/drive/MyDrive/hallucination-analyzer")

from app.services.pipeline import HallucinationPipeline

pipeline = HallucinationPipeline()

y_true = []
y_pred = []
rows = []

for item in fever_samples:
    result = pipeline.analyze(item["context"], item["answer"])

    y_true.append(item["label"])
    y_pred.append(result["final_label"])

    rows.append({
        "true_label": item["label"],
        "predicted_label": result["final_label"],
        "confidence": result["confidence"],
        "semantic_similarity": result["semantic_similarity"],
        "entailment_prob": result["nli"]["entailment_prob"],
        "original_fever_label": item["original_label"]
    })

df_fever = pd.DataFrame(rows)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

df_fever.head()


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Accuracy: 0.754

Confusion Matrix:

[[148   4]
 [119 229]]

Classification Report:

               precision    recall  f1-score   support

NOT_SUPPORTED       0.55      0.97      0.71       152
    SUPPORTED       0.98      0.66      0.79       348

     accuracy                           0.75       500
    macro avg       0.77      0.82      0.75       500
 weighted avg       0.85      0.75      0.76       500



,true_label,predicted_label,confidence,semantic_similarity,entailment_prob,original_fever_label
0,SUPPORTED,SUPPORTED,0.837547,0.576918,0.977885,SUPPORTS
1,NOT_SUPPORTED,NOT_SUPPORTED,0.207091,0.582904,0.004730,REFUTES
2,SUPPORTED,SUPPORTED,0.615635,0.668042,0.587415,SUPPORTS
3,SUPPORTED,SUPPORTED,0.914046,0.806539,0.971934,SUPPORTS
4,SUPPORTED,SUPPORTED,0.945284,0.873183,0.984107,SUPPORTS


__📌 Problem:__

Model is too biased toward NOT_SUPPORTED (high recall 0.97 for NOT_SUPPORTED)


It is catching hallucinations well, but it’s rejecting true supported claims too often.
So now we tune threshold + weights like real ML engineers 😈📈

#PHASE 10: HYPERPARAMETRIC TUNING (Weights + Threshold)

We already stored in df_fever:

entailment_prob

semantic_similarity

true_label

Now we do grid search.

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

best_f1 = -1
best_params = None
best_acc = None

results = []

for nli_weight in np.arange(0.30, 0.91, 0.05):
    sim_weight = 1 - nli_weight

    for threshold in np.arange(0.30, 0.81, 0.05):

        preds = []
        scores = []

        for _, row in df_fever.iterrows():
            score = nli_weight * row["entailment_prob"] + sim_weight * row["semantic_similarity"]
            pred = "SUPPORTED" if score >= threshold else "NOT_SUPPORTED"

            preds.append(pred)
            scores.append(score)

        acc = accuracy_score(df_fever["true_label"], preds)
        f1 = f1_score(df_fever["true_label"], preds, pos_label="SUPPORTED")

        results.append((nli_weight, sim_weight, threshold, acc, f1))

        if f1 > best_f1:
            best_f1 = f1
            best_acc = acc
            best_params = (nli_weight, sim_weight, threshold)

print("BEST PARAMS FOUND 🔥")
print("NLI_WEIGHT:", best_params[0])
print("SIM_WEIGHT:", best_params[1])
print("THRESHOLD:", best_params[2])
print("Best Accuracy:", best_acc)
print("Best F1 (SUPPORTED):", best_f1)


BEST PARAMS FOUND 🔥
NLI_WEIGHT: 0.5999999999999999
SIM_WEIGHT: 0.40000000000000013
THRESHOLD: 0.3
Best Accuracy: 0.858
Best F1 (SUPPORTED): 0.8938714499252616
